# Practical 05a — Informed Search: A-star with Heuristics (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 5 |
| **Related lectures** | Lectures 11, 12 and 13 |
| **Duration** | 3 hours |
| **Covers the same work as** | Practical 05 (same concepts, same five tasks, same marks) |
| **Submission** | This notebook, run top to bottom from a clean kernel |

This is an easier route through Practical 05. Nothing is left out. You still add a heuristic
to a search problem, turn one searcher into lowest-cost-first, greedy and A-star, check that
the heuristic never overestimates, break it on purpose, fool greedy search with a maze, and
write branch and bound. The difference is that each idea comes in a small step, most of the
code is written for you, and you fill in the missing pieces.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. When you see `___` (three underscores) in code, that is a **blank** for you to fill in.
   Replace the `___` with your answer, then run the cell again.
3. After most exercises there is a **✅ You should see** box. Compare what your cell printed
   with the box:
   - **the same**: you got it right, so move on;
   - **different**: look at your blanks again. The comment next to each blank tells you
     what goes there.
4. Some cells ask you to **predict** first. Write your guess in the blank, then run the next
   cell. It prints the real answer, so you can see whether your guess was right.
5. If you see a red error message, the usual cause is a blank you have not filled yet.
   Read the **last line** of the error, which names the problem.
6. If you restart the notebook (Kernel → Restart), run all the cells above the one you are
   on again, starting from the setup cell.

> 💡 **Last week's lab (P04a) is the starting point.** The maze, `parse` and `MazeProblem`
> are the ones you wrote there. The only new idea today is the **heuristic**.

## 1. Objectives

By the end of this session you will be able to:

1. Add a heuristic function to a search problem.
2. Implement greedy best-first search and A-star by changing only the frontier ordering.
3. Construct an admissible heuristic by relaxation and justify it.
4. Measure the effect of a heuristic on nodes expanded.
5. Demonstrate what happens when a heuristic overestimates.

## 2. Background — the idea in plain words

**Last week the search had no idea where the goal was.** Breadth-first search spread out
evenly in every direction, like water poured on the floor. Today we give the search a
**hint**, a rough guess of how far each cell is from the goal, so it can head the right way.

**An everyday example.** You drive from Islamabad to Lahore. At a town on the way, two
numbers matter:

| Number | Meaning | In the car | Fact or guess? |
|---|---|---|---|
| **g** | cost **already spent** to get here | km on the trip meter | a **fact**, you have driven it |
| **h** | **estimate** of the cost still to go | the straight-line distance to Lahore on a map | a **guess**, the road may bend |
| **f = g + h** | estimate of the **whole trip** through this town | trip meter + straight-line distance | a **guess** |

The straight-line distance can **never be longer** than the real road, because no road is
shorter than a straight line. A guess that never overestimates is called **admissible**.

**Four search strategies, one difference (Lectures 11–13).** All four take the path with
the **smallest priority** off the frontier. They differ **only** in how they score a path:

| Strategy | Priority (smallest first) | Thinks about | Shortest route guaranteed? |
|---|---|---|---|
| **Lowest-cost-first** (UCS) | `g` | only the past | **yes** |
| **Greedy best-first** | `h` | only the future (a guess) | **no** |
| **A-star** | `g + h` | both | **yes**, if `h` is admissible |
| **Weighted A-star** | `g + w × h`, with `w > 1` | the guess counts extra | **no** |

**Where does a good h come from? Relaxation (Lecture 12).** Make the problem easier by
**deleting a rule**, then solve the easy problem exactly. In a maze, delete the rule "you
cannot walk through walls". With no walls the shortest route from `(r, c)` to the goal
`(gr, gc)` is just the **grid distance** (also called the **Manhattan distance**):

```
h = |r − gr| + |c − gc|        (rows to go + columns to go)
```

Any real route also works when the walls are gone, so the real route can never be shorter
than this number. That is why it never overestimates.

**Words used in this lab:**

| Word | What it means here |
|---|---|
| **heuristic** `h` | a function that guesses the remaining cost from a cell to the goal |
| **admissible** | `h` is never bigger than the true remaining cost, for any cell |
| **priority queue** | a list that always hands back its **smallest** item first (Python's `heapq`) |
| **nodes expanded** | how many cells the search took off the frontier and looked at: our measure of **work** |
| **solution cost** | the number of steps in the route that was returned |
| **optimal** | the returned route is the shortest one there is |

**Key points to keep in mind while you work:**

- g is a fact about the path taken; h is a guess about territory not yet explored.
- An admissible heuristic never overestimates, which is what lets A-star stop safely at a goal.
- Overestimating makes the search faster and the answers wrong, with no error raised.
- The zero heuristic is admissible and useless: it turns A-star back into lowest-cost-first.

## 3. Setup

**Software:** Python 3.10 or later, and `heapq` from the standard library. Nothing to install.

Run the cell below **first, every time you open the notebook.**

In [1]:
import heapq                  # a priority queue: always gives back the smallest item
from itertools import count   # a ticket machine: 0, 1, 2, 3, ...

___ = None                    # the blank you will replace with your answers
print('Setup done. You are ready to start.')

Setup done. You are ready to start.


## 4. Python warm-up — only what you need today

Five small Python ideas are used in this lab. Each one has a demo cell (just run it) and a
tiny exercise.

### Warm-up 1 — abs() and the grid distance

`abs(x)` drops the minus sign: `abs(-3)` is `3`. The **grid distance** between two cells
counts the rows between them plus the columns between them. We use `abs` because we do not
care whether the goal is above or below, left or right.

In [2]:
a = (1, 1)            # (row, col)
b = (4, 3)
rows_apart = abs(a[0] - b[0])      # |1 - 4| = 3
cols_apart = abs(a[1] - b[1])      # |1 - 3| = 2
print('rows apart:', rows_apart, '  cols apart:', cols_apart, '  grid distance:', rows_apart + cols_apart)

rows apart: 3   cols apart: 2   grid distance: 5


**Your turn:** work these out by hand (no code), then run the cell to compare.

In [3]:
dist_1 = 24     # grid distance from (1, 1) to (13, 13)
dist_2 = 11     # grid distance from (5, 10) to (13, 13)
dist_3 = 0      # grid distance from (13, 13) to (13, 13)

print('your answers :', dist_1, dist_2, dist_3)
print('real answers :', abs(1-13) + abs(1-13), abs(5-13) + abs(10-13), abs(13-13) + abs(13-13))

your answers : 24 11 0
real answers : 24 11 0


**✅ You should see:**

```
your answers : 24 11 0
real answers : 24 11 0
```

### Warm-up 2 — heapq: a list that always gives back the smallest item

`heapq.heappush(h, x)` puts `x` in; `heapq.heappop(h)` takes out the **smallest** item, no
matter in which order things went in. A search can then always work on the path with the
smallest priority.

In [4]:
h = []
for x in [5, 2, 8, 1]:
    heapq.heappush(h, x)

print(heapq.heappop(h))     # 1  - the smallest, although it went in last
print(heapq.heappop(h))     # 2

1
2


**Your turn — predict:** the heap below gets `7, 3, 9, 4`. What do the first two pops give?

In [5]:
first_pop  = 3
second_pop = 4

In [6]:
# Run this to reveal the real answer.
h = []
for x in [7, 3, 9, 4]:
    heapq.heappush(h, x)
print('you said:', first_pop, second_pop, '   real:', heapq.heappop(h), heapq.heappop(h))

you said: 3 4    real: 3 4


### Warm-up 3 — Tuples are compared item by item, and the ticket breaks ties

The search pushes **tuples** such as `(priority, ticket, path)`. Python compares two tuples
by their **first** items; only if those are **equal** does it look at the **second** item.

The **ticket** comes from `count()`, which hands out 0, 1, 2, 3, … like a token machine at
the bank. When two paths have the **same priority**, the one with the smaller ticket, the
one that arrived **first**, comes out first. (Remember this for Task 3!)

In [7]:
ticket = count()
h = []
heapq.heappush(h, (4, next(ticket), 'A'))    # priority 4, ticket 0
heapq.heappush(h, (2, next(ticket), 'B'))    # priority 2, ticket 1
heapq.heappush(h, (2, next(ticket), 'C'))    # priority 2, ticket 2  - same priority as B

print(heapq.heappop(h))    # (2, 1, 'B') - smallest priority; B and C tie, B has the smaller ticket

(2, 1, 'B')


**Your turn — predict:** what comes out of the heap **next**, and after that?

In [8]:
second_out = 'C'     # 'A', 'B' or 'C'
third_out  = 'A'

In [9]:
# Run this to reveal the real answer.
print('you said:', second_out, third_out, '   real:', heapq.heappop(h)[2], heapq.heappop(h)[2])

you said: C A    real: C A


### Warm-up 4 — A function can be passed to another function

In Python a function is a value, like a number. You can hand it to another function, which
then calls it. Today we hand different **heuristics** to the same search.

In [10]:
def double(x):
    return 2 * x

def apply_twice(f, x):
    return f(f(x))          # call whichever function we were given, twice

print(apply_twice(double, 5))      # double(double(5)) = 20

20


**Your turn:** write `zero(x)` so that it **always** returns `0`, whatever `x` is.

In [11]:
def zero(x):
    return 0

print(apply_twice(zero, 5), apply_twice(double, 3))

0 12


**✅ You should see:**

```
0 12
```

### Warm-up 5 — A function that calls itself (recursion)

Branch and bound (Task 5) goes **deep first** by calling itself. Each call goes one step
deeper; when a call finishes, Python comes back to the call that made it.

Below, `best` is a **dictionary**. A function inside another function can change the
**contents** of a dictionary from outside, which is how a deep call reports "I found
something better" to everyone else.

In [12]:
best = {'smallest': 999}

def walk_down(n):
    print('  ' * (6 - n) + 'at', n)      # indent more the deeper we are
    if n < best['smallest']:
        best['smallest'] = n             # change the dict: everyone sees the new value
    if n > 0:
        walk_down(n - 2)                 # the function calls ITSELF, one level deeper

walk_down(6)
print('smallest number seen:', best['smallest'])

at 6
    at 4
        at 2
            at 0
smallest number seen: 0


**Your turn — predict:** what would `walk_down(4)` print as the **last** `at` line, and what is the smallest number seen?

In [13]:
last_at  = 0
smallest = 0

In [14]:
# Run this to reveal the real answer.
best = {'smallest': 999}
walk_down(4)
print('you said:', last_at, smallest, '   real smallest:', best['smallest'])

    at 4
        at 2
            at 0
you said: 0 0    real smallest: 0


## 5. Procedure — build informed search one piece at a time

Each step introduces **one** idea. Run the demo, read the explanation, then do the small
exercise.

### Step 1 — The maze, and the heuristic

This is the **15 × 15 maze with loops** that you built in P04a, so there are several routes
to the goal. `parse` and `MazeProblem` are exactly as you wrote them last week.

The only new part is `manhattan`, the heuristic. **Fill in its one blank**: rows to go plus
columns to go (warm-up 1).

In [15]:
MAZE = [
    '###############',
    '#S............#',
    '#.###.#.#####.#',
    '#.#...#.....#.#',
    '#.#.#######.#.#',
    '#...#.......#.#',
    '###.#.###.#.#.#',
    '#...#...#...#.#',
    '#.#####.#####.#',
    '#.............#',
    '#.###.#####.###',
    '#.#.........#.#',
    '#.#..######.#.#',
    '#...#........G#',
    '###############',
]

def parse(maze):
    walls, start, goal = set(), None, None
    for r, row in enumerate(maze):
        for c, ch in enumerate(row):
            if ch == '#':   walls.add((r, c))
            elif ch == 'S': start = (r, c)
            elif ch == 'G': goal = (r, c)
    return walls, start, goal

class MazeProblem:
    def __init__(self, walls, start, goal):
        self.walls, self.start, self.goal = walls, start, goal

    def start_state(self):
        return self.start

    def is_goal(self, state):
        return state == self.goal

    def actions(self, state):
        """Yield (next_state, cost) for each legal move: up, down, left, right."""
        r, c = state
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (r + dr, c + dc)
            if nxt not in self.walls:
                yield nxt, 1

def manhattan(state, goal):
    """RELAXATION: pretend the walls are not there. The shortest route is then
    rows to go + columns to go, and no real route can be shorter than that."""
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

WALLS, START, GOAL = parse(MAZE)
problem = MazeProblem(WALLS, START, GOAL)
print('start', START, '  goal', GOAL)
print('h at start:', manhattan(START, GOAL), ' (the true cost is at least this)')

start (1, 1)   goal (13, 13)
h at start: 24  (the true cost is at least this)


**✅ You should see:**

```
start (1, 1)   goal (13, 13)
h at start: 24  (the true cost is at least this)
```

### Step 2 — g, h and f by hand

Below are three cells of the maze, each with the **g** of the cheapest way to reach it
from `S` (you can count it on the maze above). Work out **h** (grid distance to the goal
`(13, 13)`) and **f = g + h** for each one.

| Cell | How you get there from S | g |
|---|---|---|
| A = `(1, 5)` | 4 steps right along the top row | 4 |
| B = `(3, 1)` | 2 steps down | 2 |
| C = `(7, 1)` | down the left side, then back up to the left wall | 10 |

In [16]:
h_A = 20          # |1 - 13| + |5 - 13|
f_A = 24          # 4 + h_A
h_B = 22
f_B = 24
h_C = 18
f_C = 28

expand_first = ['A', 'B']   # A-star expands the SMALLEST f first. Which cell(s)?

In [17]:
# Run this to reveal the real values.
print('%-4s %-8s %3s %3s %3s' % ('', 'cell', 'g', 'h', 'f'))
for name, cell, g in [('A', (1, 5), 4), ('B', (3, 1), 2), ('C', (7, 1), 10)]:
    h = manhattan(cell, GOAL)
    print('%-4s %-8s %3d %3d %3d' % (name, cell, g, h, g + h))

     cell       g   h   f
A    (1, 5)     4  20  24
B    (3, 1)     2  22  24
C    (7, 1)    10  18  28


Notice that **f stays at 24** for cells on a route heading towards the goal (A and B), and
**rises** for C, a route that wandered away. A-star leaves C until later. This is how it
avoids wasting work: a path that has already wandered pays for it in **g**.

### Step 3 — One searcher, four strategies

This is last week's search with **two** changes:

1. the frontier is a **priority queue** (`heapq`) instead of a plain list, so the path with
   the **smallest priority** always comes out first;
2. a small function `priority(g, state)` scores each path, and this is the **only** place
   where the strategies differ.

Each frontier item is a tuple `(priority, ticket, g, path)`: the score first (so it decides
the order), then the ticket (it breaks ties, warm-up 3), then the cost so far, then the path.

**Fill in the five blanks.** The background table in Section 2 tells you blanks 1–3.

The search returns **four** things: the `path`, its `cost`, how many nodes it `expanded`,
and the set of cells it `explored` (used in Step 4 to draw a picture).

In [18]:
def search(problem, strategy, heuristic=manhattan, weight=1.0):
    """strategy: 'ucs' | 'greedy' | 'astar'.   weight multiplies h (A-star only)."""
    ticket = count()
    start = problem.start_state()

    def priority(g, state):
        h = heuristic(state, problem.goal)
        if strategy == 'ucs':
            return g                      # blank 1: cost so far ONLY
        if strategy == 'greedy':
            return h                      # blank 2: estimate ONLY
        return g + weight * h             # blank 3: A-star: cost so far + weight x estimate

    frontier = [(priority(0, start), next(ticket), 0, [start])]
    visited, expanded = set(), 0
    while frontier:
        _, _, g, path = heapq.heappop(frontier)     # blank 4: take the SMALLEST priority
        state = path[-1]
        if state in visited:
            continue                        # already expanded: skip it
        visited.add(state)
        expanded += 1
        if problem.is_goal(state):
            return path, g, expanded, visited
        for nxt, cost in problem.actions(state):
            if nxt not in visited:
                ng = g + cost             # blank 5: the cost of the path one step longer
                heapq.heappush(frontier, (priority(ng, nxt), next(ticket), ng, path + [nxt]))
    return None, None, expanded, visited

print('%-7s %6s %16s' % ('', 'cost', 'nodes expanded'))
for strat in ['ucs', 'greedy', 'astar']:
    path, cost, expanded, explored = search(problem, strat)
    print('%-7s %6d %16d' % (strat, cost, expanded))

          cost   nodes expanded
ucs         28               98
greedy      32               33
astar       28               69


**✅ You should see:**

```
          cost   nodes expanded
ucs         28               98
greedy      32               33
astar       28               69
```

Read the two columns **together**:

- **UCS** and **A-star** both find the shortest route (cost 28), but A-star needs far
  fewer nodes. The heuristic **saved work without hurting the answer**.
- **Greedy** is the fastest of all, but its route costs **32**, not 28. It is fast **and wrong**.

### Step 4 — Seeing the work

`show` draws the maze with the route as `*` and every **other** cell the search expanded as
`o`. Fewer `o` means less work. Just run it.

In [19]:
def show(maze, path, explored=(), title=''):
    grid = [list(row) for row in maze]
    for (r, c) in explored:
        if grid[r][c] == '.':
            grid[r][c] = 'o'
    for (r, c) in path[1:-1]:
        grid[r][c] = '*'
    print(title)
    print('\n'.join(''.join(row) for row in grid))
    print()

for strat in ['ucs', 'astar', 'greedy']:
    path, cost, expanded, explored = search(problem, strat)
    show(MAZE, path, explored, '%s: cost %d, %d nodes expanded' % (strat.upper(), cost, expanded))

UCS: cost 28, 98 nodes expanded
###############
#Soooooooooooo#
#*###o#o#####o#
#*#ooo#ooooo#o#
#*#o#######o#o#
#***#ooooooo#o#
###*#o###o#o#o#
#***#ooo#ooo#o#
#*#####o#####o#
#*****oooooooo#
#o###*#####o###
#o#oo*******#.#
#o#oo######*#.#
#ooo#....oo**G#
###############

ASTAR: cost 28, 69 nodes expanded
###############
#S************#
#o###o#o#####*#
#o#ooo#ooooo#*#
#o#o#######o#*#
#ooo#....ooo#*#
###o#.###o#o#*#
#ooo#...#ooo#*#
#o#####.#####*#
#ooooo.....***#
#o###.#####*###
#o#........*#.#
#o#..######*#.#
#o..#......**G#
###############

GREEDY: cost 32, 33 nodes expanded
###############
#S............#
#*###.#.#####.#
#*#...#.....#.#
#*#.#######.#.#
#***#.......#.#
###*#.###.#.#.#
#***#...#...#.#
#*#####.#####.#
#*............#
#*###.#####.###
#*#.********#.#
#*#**######*#.#
#***#......**G#
###############



**Look at the three pictures:**

- **UCS** explores almost the **whole** maze: it has no idea where the goal is.
- **A-star** leaves many cells in the **middle and bottom** alone (the `.` cells), because
  a route through them has wandered away from the goal, so their `f` is large.
- **Greedy** explores very little, but look at its route. It runs straight down the left side,
  because every step down lowers `h`. At the bottom it hits a wall and has to climb back up to
  row 11 to get round it: a **detour** of 4 extra steps. It only looks at `h`, so it never
  noticed that the top corridor was shorter.

### Step 5 — Checking admissibility by hand

Admissibility is something to **check**, not to assume. `true_cost_from` finds the real
shortest distance from a cell to the goal (by running UCS from that cell). For each cell,
the estimate `h` must be **no bigger** than the true cost.

**Fill in the one blank:** `admissible` should be `True` when `h` is less than or equal to
`true`.

In [20]:
def true_cost_from(cell):
    sub = MazeProblem(WALLS, cell, GOAL)
    path, cost, expanded, explored = search(sub, 'ucs')
    return cost

print('%-10s %5s %6s  %s' % ('cell', 'h', 'true', 'admissible?'))
for cell in [START, (1, 5), (5, 3), (7, 7), (9, 2), (11, 9)]:
    h = manhattan(cell, GOAL)
    true = true_cost_from(cell)
    admissible = h <= true                  # blank: True if h does not overestimate
    print('%-10s %5d %6d  %s' % (cell, h, true, admissible))

cell           h   true  admissible?
(1, 1)        24     28  True
(1, 5)        20     24  True
(5, 3)        18     22  True
(7, 7)        12     12  True
(9, 2)        15     15  True
(11, 9)        6      6  True


**✅ You should see:**

```
cell           h   true  admissible?
(1, 1)        24     28  True
(1, 5)        20     24  True
(5, 3)        18     22  True
(7, 7)        12     12  True
(9, 2)        15     15  True
(11, 9)        6      6  True
```

`h` is **never bigger** than the true cost. For the last three cells it is exactly right,
because from there a route with no backward steps exists. Near the start it is 4 too small,
because the walls force the real route to bend. Six cells is evidence, not proof. The **proof** is the relaxation argument in
Section 2, and you will write it out in Task 1.

### Step 6 — Breaking admissibility on purpose

Now we multiply the heuristic by a **weight** `w`. With `w > 1` the heuristic **can**
overestimate, so A-star's guarantee is gone. Watch two columns at once.

**Fill in the one blank** so each search uses the weight `w` of the loop.

In [21]:
print('%-8s %6s %10s' % ('weight', 'cost', 'expanded'))
for w in [1.0, 1.5, 2.0, 3.0, 5.0]:
    path, cost, expanded, explored = search(problem, 'astar', weight=w)
    print('%-8.1f %6d %10d' % (w, cost, expanded))

weight     cost   expanded
1.0          28         69
1.5          28         52
2.0          28         38
3.0          32         34
5.0          32         33


**✅ You should see:**

```
weight     cost   expanded
1.0          28         69
1.5          28         52
2.0          28         38
3.0          32         34
5.0          32         33
```

As the weight rises, the search gets **cheaper** (69 → 33 nodes). At weight 3 the route
gets **worse** (28 → 32). **No error is raised.** The program prints a route that looks
perfectly fine. It is just not the shortest. This is the **silent failure** from Lecture 11.

## 6. Tasks (marked)

These are the same five tasks as Practical 05, worth the same marks. Each one is split into
small parts.

### Task 1 — Justify your heuristic in four sentences

> **Marks:** 4

Write the four justification sentences from Lecture 12 for the grid-distance heuristic.
**Double-click this cell to edit it**, and replace each `________`. Section 2 (Background)
has everything you need.

1. **Which constraint was relaxed?** I deleted the rule that **you cannot walk through walls**.

2. **What does the relaxed problem cost?** With that rule gone, the cheapest route from
   `(r, c)` to the goal costs exactly **|r − gr| + |c − gc| (rows to go plus columns to go)**, which is the grid distance.

3. **Why can it not overestimate?** Every route in the real maze is also a route in the
   relaxed problem, because **every move that is legal with walls is still legal when the walls are removed, so a real route can still be walked in the relaxed maze**. So the real cheapest route can never be **shorter** than
   the relaxed one.

4. **Is it useful?** **Yes**, because **it is much larger than `h = 0` (24 instead of 0 at the start) while still never overestimating, so it steers A-star towards the goal: 69 nodes expanded instead of 98, with the same optimal cost of 28.**

### Task 2 — A second, weaker heuristic

> **Marks:** 4

A heuristic can be admissible but **weak**: it never overestimates, but it is a poor guess.
We try two:

| Heuristic | Formula | Idea |
|---|---|---|
| `row_only` | `abs(row − goal row)` | forget the columns completely |
| `zero_h` | `0` | no information at all |

**Part A — write them.** Fill in the **two** blanks (warm-up 1 and warm-up 4 help).

In [22]:
def row_only(state, goal):
    return abs(state[0] - goal[0])        # blank 1: rows to go, columns ignored

def zero_h(state, goal):
    return 0                              # blank 2: always zero

print('at the start:  manhattan', manhattan(START, GOAL),
      '  row_only', row_only(START, GOAL), '  zero_h', zero_h(START, GOAL))

at the start:  manhattan 24   row_only 12   zero_h 0


**✅ You should see:**

```
at the start:  manhattan 24   row_only 12   zero_h 0
```

**Part B — is row_only still admissible?** This cell checks **every** open cell of the maze
and counts the cells where `row_only` is bigger than the true cost. Just run it.

In [23]:
open_cells = [(r, c) for r in range(len(MAZE)) for c in range(len(MAZE[0])) if (r, c) not in WALLS]
too_big = [cell for cell in open_cells if row_only(cell, GOAL) > true_cost_from(cell)]
print('open cells checked:', len(open_cells))
print('cells where row_only overestimates:', len(too_big))

open cells checked: 104
cells where row_only overestimates: 0


**✅ You should see:**

```
open cells checked: 104
cells where row_only overestimates: 0
```

**Part C — compare.** Fill in the **one** blank so each search uses the heuristic `h_func` from the loop.

In [24]:
print('%-12s %12s %10s %6s' % ('heuristic', 'h at start', 'expanded', 'cost'))
t2 = {}
for name, h_func in [('zero_h', zero_h), ('row_only', row_only), ('manhattan', manhattan)]:
    path, cost, expanded, explored = search(problem, 'astar', heuristic=h_func)
    t2[name] = (expanded, cost)
    print('%-12s %12d %10d %6d' % (name, h_func(START, GOAL), expanded, cost))

heuristic      h at start   expanded   cost
zero_h                  0         98     28
row_only               12         97     28
manhattan              24         69     28


**✅ You should see:**

```
heuristic      h at start   expanded   cost
zero_h                  0         98     28
row_only               12         97     28
manhattan              24         69     28
```

**Part D — explain (double-click to edit).** Replace each `________`.

All three heuristics returned cost **28**, because **all three are admissible (none overestimates), so A-star is guaranteed to return the shortest route**. They differ only in the
number of nodes expanded: `zero_h` **98**, `row_only` **97**, `manhattan` **69**.

`zero_h` expanded exactly as many nodes as lowest-cost-first in Step 3, because with
`h = 0` the priority `g + h` is just **g**.

`row_only` saved almost nothing, because it gives the same `h` to every cell in the
same **row**, however far apart they are. A bigger `h` that still never overestimates is
better, because **it is closer to the true remaining cost, so `f = g + h` rises sooner along routes that wander away from the goal, and A-star can leave those routes unexpanded.**

### Task 3 — Make greedy search fail

> **Marks:** 5

You need a maze where greedy search returns a route **noticeably longer** than A-star's.
Aim for **at least 1.5 times** as long.

**Part A — the tie.** In the small maze below, `S` is at `(1, 1)`. Suppose the goal is
somewhere down and to the right. From `S`, a step **down** and a step **right** both lower `h`
by 1, so greedy sees them as **equally good**: a tie. Our `actions` tries moves in the
order up, down, left, right, so the **down** path is pushed first and gets the **smaller
ticket**. With a tie in priority, which path does the heap hand back first (warm-up 3)?

In [25]:
greedy_goes_first = 'down'      # 'down' or 'right'

**Part B — set the trap.** `TRAP_NO_GOAL` has a **short way** along the top corridor and a
**long, winding way** through the bottom half. There is no goal yet. Choose where to put
`G` so that greedy, going **down** first, wanders the long way while A-star takes the short
one.

> **Hint:** put `G` in the **right-hand corridor**, a few rows down from the top. Try a spot,
> run the cell, read the last line, and try another spot if the ratio is below 1.5. Each try
> takes a second.

In [26]:
TRAP_NO_GOAL = [
    '###########',
    '#S........#',
    '#.#######.#',
    '#.#.....#.#',
    '#.#.###.#.#',
    '#...#...#.#',
    '#####.###.#',
    '#.........#',
    '###########',
]

def place_goal(maze, spot):
    """Return a copy of maze with 'G' written at spot = (row, col)."""
    r, c = spot
    rows = list(maze)
    rows[r] = rows[r][:c] + 'G' + rows[r][c + 1:]
    return rows

GOAL_SPOT = (5, 9)          # (row, col) of your goal - it must be a '.' cell

TRAP_MAZE = place_goal(TRAP_NO_GOAL, GOAL_SPOT)
trap = MazeProblem(*parse(TRAP_MAZE))

t3 = {}
for strat in ['astar', 'greedy']:
    path, cost, expanded, explored = search(trap, strat)
    t3[strat] = (cost, expanded)
    show(TRAP_MAZE, path, (), '%s: cost %d, %d nodes expanded' % (strat.upper(), cost, expanded))

print('greedy route is %.1f times as long as the A-star route' % (t3['greedy'][0] / t3['astar'][0]))

ASTAR: cost 12, 19 nodes expanded
###########
#S********#
#.#######*#
#.#.....#*#
#.#.###.#*#
#...#...#G#
#####.###.#
#.........#
###########

GREEDY: cost 24, 25 nodes expanded
###########
#S........#
#*#######.#
#*#*****#.#
#*#*###*#.#
#***#***#G#
#####*###*#
#....*****#
###########

greedy route is 2.0 times as long as the A-star route


**✅ Target:** the last line should say **at least 1.5 times**. (Several goal spots work.)

**Part C — explain (double-click to edit).** Replace each `________`.

Greedy's route cost **24** and A-star's cost **12**, so greedy's was **2.0** times as long.

At the start, going down and going right had the same `h`, so greedy took **down** because
**the down path was pushed first, so it has the smaller ticket and wins the tie**. After that, every cell on the
long way round had a **smaller** `h` than the first
cell of the short top corridor, `(1, 2)`, whose `h` is **11**. Greedy always takes the
smallest `h`, so it never went back to the top corridor.

A-star was not fooled because it adds **g**, the cost already spent, so a route that has
wandered for many steps **pays for every step in `f`, until its `f` rises above that of the short top corridor and A-star switches back to it.**

### Task 4 — Quantify the silent failure

> **Marks:** 4

Use weighted A-star on **your trap maze** from Task 3. First find the true optimal cost
(weight 1, which is admissible). Then try bigger weights and mark each row "yes" if the
cost is still optimal.

**Fill in the two blanks.**

In [27]:
best_path, optimal_cost, best_expanded, _ = search(trap, 'astar', weight=1.0)   # blank 1: admissible
print('optimal cost:', optimal_cost)
print()

print('%-8s %6s %10s %10s' % ('weight', 'cost', 'expanded', 'optimal?'))
t4 = []
for w in [1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0]:
    path, cost, expanded, explored = search(trap, 'astar', weight=w)
    still_optimal = cost <= optimal_cost            # blank 2: True if no worse than the optimum
    t4.append((w, cost, expanded, still_optimal))
    print('%-8.1f %6d %10d %10s' % (w, cost, expanded, 'yes' if still_optimal else 'NO'))

first_bad = [w for (w, cost, expanded, ok) in t4 if not ok]
print()
print('smallest weight tried that lost optimality:', first_bad[0] if first_bad else 'none')

optimal cost: 12

weight     cost   expanded   optimal?
1.0          12         19        yes
1.5          12         19        yes
2.0          12         20        yes
3.0          12         30        yes
5.0          24         25         NO
7.0          24         25         NO
10.0         24         25         NO

smallest weight tried that lost optimality: 5.0


**✅ Target:** at least one weight should print `NO`. (Every goal spot that reached 1.5 in
Task 3 does so by weight 10.)

**Write down (double-click to edit):**

The smallest weight at which the route stopped being optimal was **5.0** (cost 24 instead of 12).

As the weight went up, the number of nodes expanded **went up at first (19 → 20 → 30 at weight 3) and then dropped back to 25 once the route became wrong, never going below the 19 of weight 1** (went down / went up / went down
and then up). Compare with Step 6 on the big maze: there it **went down at every weight (69 → 52 → 38 → 34 → 33)**.

### Task 5 — Branch and bound

> **Marks:** 5

**Depth-first branch and bound** (Lecture 13) searches **depth-first** (one route all the way
to the end, then backs up) and keeps two things:

- the **best complete route found so far**, whose cost is the **bound**. At the start the
  bound is infinity (`float('inf')`), so the first route found always becomes the best;
- a **pruning rule**: before going deeper into a cell, work out `g + h`. Because `h` never
  overestimates, `g + h` is the **least** this route could possibly cost. If even that is
  not better than the bound, nothing down this branch can win, so we **cut it off** (prune).

Each time a better route is found the bound **tightens** (gets smaller) and is printed.

**Part A — fill in the three blanks** (warm-up 5 shows the dictionary and recursion ideas).

In [28]:
def branch_and_bound(problem, heuristic=manhattan):
    best = {'cost': float('inf'), 'path': None}     # the bound starts at infinity
    counts = {'expanded': 0, 'pruned': 0}

    def dive(path, g):
        state = path[-1]
        counts['expanded'] += 1
        if problem.is_goal(state):
            if g < best['cost']:                   # blank 1: better than the best so far?
                best['cost'] = g                   # blank 2: tighten the bound
                best['path'] = list(path)
                print('  new bound:', best['cost'])
            return
        for nxt, cost in problem.actions(state):
            if nxt in path:
                continue                             # never walk back over our own route
            ng = g + cost
            optimistic = ng + heuristic(nxt, problem.goal)   # blank 3: the least this route can cost
            if optimistic >= best['cost']:
                counts['pruned'] += 1                # cannot beat the best: cut it off
                continue
            dive(path + [nxt], ng)                   # go one step deeper

    dive([problem.start_state()], 0)
    return best['path'], best['cost'], counts['expanded'], counts['pruned']

print('branch and bound on the big maze:')
bb_path, bb_cost, bb_expanded, bb_pruned = branch_and_bound(problem)
print('final cost', bb_cost, '  nodes expanded', bb_expanded, '  branches pruned', bb_pruned)

branch and bound on the big maze:
  new bound: 52
  new bound: 48
  new bound: 44
  new bound: 36
  new bound: 32
  new bound: 28
final cost 28   nodes expanded 272   branches pruned 24


**✅ You should see:**

```
branch and bound on the big maze:
  new bound: 52
  new bound: 48
  new bound: 44
  new bound: 36
  new bound: 32
  new bound: 28
final cost 28   nodes expanded 272   branches pruned 24
```

**Part B — compare with the others.** Just run it.

In [29]:
print('%-24s %6s %10s' % ('method', 'cost', 'expanded'))
for name, strat in [('lowest-cost-first', 'ucs'), ('A-star', 'astar')]:
    path, cost, expanded, explored = search(problem, strat)
    print('%-24s %6d %10d' % (name, cost, expanded))
print('%-24s %6d %10d' % ('branch and bound (DFS)', bb_cost, bb_expanded))

method                     cost   expanded
lowest-cost-first            28         98
A-star                       28         69
branch and bound (DFS)       28        272


**✅ You should see:**

```
method                     cost   expanded
lowest-cost-first            28         98
A-star                       28         69
branch and bound (DFS)       28        272
```

**Part C — explain (double-click to edit).** Replace each `________`.

The bound tightened **6** times: **52** → … → **28**. The final cost, **28**, is the
same as A-star's, so branch and bound finds the **optimal (shortest)** route.

It expanded **272** nodes, more than A-star, because depth-first search may visit the same
cell **many times, once for every different route that reaches it**. What it saves is **memory**: it holds only **the one route it is currently exploring** at a time, while A-star keeps
the whole **frontier (and the visited set)**.

Pruning is safe only because `h` **never overestimates**. If `h` overestimated, **`g + h` could be larger than the true cost of a branch that really would beat the bound, so that branch would be pruned by mistake and a worse route would be kept, with no error raised.**

## 7. Observations

Complete these tables from **your own** results. Do not leave them blank. The observations
carry marks and are what the viva is based on.

**On the big maze (Step 3 and Task 2)**

| Strategy | Heuristic used | Nodes expanded | Solution cost | Optimal? |
|---|---|---|---|---|
| Lowest-cost-first | none (h = 0) | 98 | 28 | Yes |
| Greedy | grid distance | 33 | 32 | No |
| A-star | grid distance | 69 | 28 | Yes |
| A-star | weaker heuristic (row only) | 97 | 28 | Yes |

**Weighted A-star (Task 4, on my trap maze; goal at (5, 9))**

| Weight | Solution cost | Nodes expanded | Still optimal? |
|---|---|---|---|
| 1.0 | 12 | 19 | Yes |
| 1.5 | 12 | 19 | Yes |
| 2.0 | 12 | 20 | Yes |
| 3.0 | 12 | 30 | Yes |
| 5.0 | 24 | 25 | No |
| 7.0 | 24 | 25 | No |
| 10.0 | 24 | 25 | No |

**Smallest weight at which the route stopped being optimal:** **5.0**

**Branch and bound (Task 5, on the big maze)**

| Bounds printed, in order | Final cost | Nodes expanded | Branches pruned |
|---|---|---|---|
| 52, 48, 44, 36, 32, 28 | 28 | 272 | 24 |

## 8. Discussion

Answer in your own words. Two or three sentences each is enough. Sentence starters are
given to help, and you may change them.

1. How many nodes did A-star save compared with lowest-cost-first, as a percentage?
2. Your weaker heuristic was still admissible. Why did it expand more nodes than the stronger one?
3. When you multiplied the heuristic, the search got faster and the answer got worse, with no
   error. Why is a silent failure more dangerous than a crash?

*Your answers:*

1. Lowest-cost-first expanded **98** nodes and A-star **69**, so A-star saved **29** nodes,
   which is **29** ÷ **98** × 100 = **29.6** %.

2. Admissible only means `h` never **overestimates**. It does not mean `h` is a **good (tight)** guess. The
   row-only heuristic **ignores the columns, so it is only 12 at the start (the true cost is 28) and is identical for every cell in a row. Because it is so far below the true cost, `f` barely rises on routes that wander, so A-star could not skip them and expanded 97 nodes, almost as many as the 98 of lowest-cost-first, against 69 for grid distance.**

3. A crash tells you **that something is wrong, immediately, so you fix it before trusting any result**. A silent failure **returns a normal-looking answer that is actually worse (cost 32 instead of 28, or 24 instead of 12), so nobody notices and the bad result keeps being used.**

## 9. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not
whether you can recite the code.

1. What do g, h and f each mean, in plain words?
2. Which constraint did you relax to obtain your heuristic?
3. Give the one-sentence argument that your heuristic never overestimates.
4. Show me the line that turns this searcher into greedy best-first.
5. What happened when you multiplied the heuristic by three, and why was there no error?
6. If h returns zero everywhere, which algorithm have you built?

## 10. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every `___` blank is filled in, and your outputs match the **✅ You should see** boxes
- [ ] Task 3: greedy's route is at least 1.5 times as long as A-star's on your trap maze
- [ ] Task 4: at least one weight prints `NO`
- [ ] The observations tables are filled in with your actual results
- [ ] Task 1, the Part C/D explanations and the discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P05a_<your-roll-number>.ipynb`

In [1]:
NAME = "Roha "
ROLL_NUMBER = "2025-cse-024"
print(NAME, ROLL_NUMBER)

Roha  2025-cse-024
